#### Import des librairies

In [1]:
import pandas as pd
from tqdm import tqdm
import numpy as np

In [2]:
path_file = '~/Bureau/exports/20241112/AGS_20241112_exports_agronomes_-archive/AGS_20241112_exports_agronomes_assolees_synthetisees.csv'

ENTREPOT_PATH = '~/Bureau/utils/data/'
DIRODUR_FILES_PATH = '~/Bureau/projets/DIRODUR/magasin/'
df = {}

In [3]:
df['assolees_synthetisees'] = pd.read_csv(path_file, sep = '@')

#### Import des données

In [4]:
# -------------------------- #
# IMPORT DES DONNÉES DIRODUR #
# -------------------------- #
dirodur_files = ['correspondance_destination_gcpe']

for file in dirodur_files:
    df[file] = pd.read_csv(DIRODUR_FILES_PATH + file+'.csv', sep = ';')

In [5]:
# ----------------------------------------- #
# CRÉATION DU RÉFÉRENTIEL DE MATCH D'UNITÉS #
# ----------------------------------------- #
dict_unites = {
    'q/ha (humidité ramenée à la norme)' : 'Q_HA_TO_STANDARD_HUMIDITY',
    't MS/ha' : 'TONNE_MS_HA',
    't sucre/ha' : 'TONNE_SUGAR_HA',
    't/ha' : 'TONNE_HA',
    'tonne_racines_ha_16_pourc' : 'TONNE_RACINES_HA_16_POURC', 
    'q/ha' : 'Q_HA',
    'kg/m²' : 'KG_M2',
    'unité/ha' : 'UNITE_HA',
    'hl/ha' : 'HL_HA'
}
df['unite_rendement'] = pd.DataFrame.from_dict(dict_unites, orient='index', columns=['unite_agrosyst']).reset_index().rename(columns={'index':'unite_nl'})

In [6]:
# complétion du référentiel transmis par les agronomes
left = df['correspondance_destination_gcpe']
right = df['unite_rendement']
df['correspondance_destination_gcpe'] = pd.merge(left, right, left_on = 'Unité_rendement', right_on = 'unite_nl', how = 'left')

In [7]:
# ----------------------------- #
# IMPORT DES DONNÉES DATAGROSYST#
# ----------------------------- #


def import_df(df_name, path_data, sep, index_col=None):
    df[df_name] = pd.read_csv(path_data+df_name+'.csv', sep = sep, index_col=index_col, low_memory=False).replace({'\r\n': '\n'}, regex=True)

def import_dfs(df_names, path_data, sep = ',', index_col=None, verbose=False):
    for df_name in tqdm(df_names) : 
        if(verbose) :
            print(" - ", df_name)
        import_df(df_name, path_data, sep, index_col=index_col)

tables_with_id = [
    'recolte_rendement_prix', 
    'action_realise',
    'action_synthetise',
    'action_realise_agrege', 
    'action_synthetise_agrege',
    'destination_valorisation',
    'recolte_rendement_prix_restructure', 
    'composant_culture', 
    'espece', 
    'variete'
]

tables_without_id = [
    'sdc_magasin_dirodur', 
    'itk_magasin_dirodur'
]

# import des données de l'entrepôt avec la colonne 'id' en index 
import_dfs(tables_with_id, ENTREPOT_PATH, sep = ',', index_col='id', verbose=False)

# import des données du magasin
import_dfs(tables_without_id, ENTREPOT_PATH, sep = ',', verbose=False)

100%|██████████| 2/2 [00:04<00:00,  2.45s/it]


In [8]:
studied_ids = ['fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_a744177e-2ce5-4974-85c5-d0dc66c4a120',
 'fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_7c008887-5843-c0fa-dca0-e890942a798b',
 'fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_f4461d32-558e-496f-91b4-5755c69ec967',
 'fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_6bcef65d-94ef-4f28-4a48-f8fef9228afd',
 'fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_10952f42-5587-45d4-87a9-c3579378fed2',
 'fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_b01428b8-4f69-4cf9-a286-952d99f556a1',
 'fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_526ef03b-910a-46e0-ab6e-f6d7cda3294f',
 'fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_2a8800c9-dfe0-4b0f-bf37-1af2640a4936',
 'fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_b93efcf6-c2e7-4bdb-b2e2-0f73677779f1',
 'fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_880cdf58-1658-416e-944d-a01587e98c69',
 'fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_89a818aa-c2bb-4d18-b528-c353fe0453fd',
 'fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_62c95b5b-2f35-4f25-9357-47e0459e5296',
 'fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_a3759377-24d5-4b47-a18d-099186d01669',
 'fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_a47e92b9-5e0a-4a93-a3f6-022f18e85052',
 'fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_fc766d60-f5a1-425b-8730-6dc6ed4e5ec1',
 'fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_bba88194-c540-4d49-8bf0-f69242ce6fdc',
 'fr.inra.agrosyst.api.entities.action.HarvestingActionValorisation_ad7d18ae-2e31-4118-bbae-43f1d798114e'
 ]

In [9]:
df['recolte_rendement_prix_test'] = df['recolte_rendement_prix'].loc[studied_ids]
df['action_realise_test'] = df['action_realise'].loc[
    df['action_realise'].index.isin(df['recolte_rendement_prix_test']['action_id'])
]
df['destination_valorisation_test'] = df['destination_valorisation'].loc[
    df['destination_valorisation'].index.isin(df['recolte_rendement_prix_test']['destination_id'])
]
df['action_realise_agrege_test'] = df['action_realise_agrege'].loc[
    df['action_realise_agrege'].index.isin(df['action_realise_test'].index)
]
df['recolte_rendement_prix_restructure_test']= df['recolte_rendement_prix_restructure'].loc[
    df['recolte_rendement_prix_restructure'].index.isin(df['recolte_rendement_prix_test'].index)
]
df['composant_culture_test'] = df['composant_culture'].loc[
    df['composant_culture'].index.isin(df['recolte_rendement_prix_restructure_test']['composant_culture_id'])
]
df['espece_test'] = df['espece'].loc[
    df['espece'].index.isin(df['composant_culture_test']['espece_id'])
]
df['variete_test'] = df['variete'].loc[
    df['variete'].index.isin(df['composant_culture_test']['variete_id'])
]

df['correspondance_destination_gcpe_test'] = df['correspondance_destination_gcpe'].loc[
    df['correspondance_destination_gcpe']['code_destination_A'].isin(df['destination_valorisation_test']['code_destination_a'])
]

In [10]:
path='./'
df['recolte_rendement_prix_test'].to_csv(path+'recolte_rendement_prix'+'.csv')
df['action_realise_test'].to_csv(path+'action_realise'+'.csv')
df['destination_valorisation_test'].to_csv(path+'destination_valorisation'+'.csv')
df['action_realise_agrege_test'].to_csv(path+'action_realise_agrege'+'.csv')
df['recolte_rendement_prix_restructure_test'].to_csv(path+'recolte_rendement_prix_restructure'+'.csv')
df['composant_culture_test'].to_csv('composant_culture'+'.csv')
df['correspondance_destination_gcpe_test'].to_csv('correspondance_destination_gcpe'+'.csv')
df['espece_test'].to_csv('espece'+'.csv')
df['variete_test'].to_csv('variete'+'.csv')